# به نام خدا

### **06-01**

<div dir='rtl'>

به زبان ساده : EDA (تحلیل اکتشافی داده‌ها) به معنای درک داده‌ها پیش از هرگونه فرضیه‌سازی یا به‌کارگیری مدل‌های پیچیده است.

مثل اینکه مریضی اومده پیشتون و تازه می خوای بفهمی که چشه.

- **7 مرحله اصلی EDA:**
1. Descriptive Statistics کل داده چه آماری داره؟
2. Data Distribution Analysis تک تک هر داده خصوصا داده هدف چه توزیعی دارد
3. Correlation Analysis ارتباط بین دو یا چند متغیر چجوریاس؟ مخالف یا وابسته به هم اند یا هیچ تاثیری رو هم ندارن؟
4. Grouping and Aggregation
5. Time Series Analysis (if applicable)
6. Handling Missing Data and Outliers
7. Feature Engineering Insights

- **چرخه کاری معمول در علم داده: (عقب و جلو زیاد داره)**
1. Problem Definition
2. Data Collection
3. Data Cleaning and Preparation
4. Exploratory Data Analysis
5. Feature Engineering
6. Modeling
7. Evaluation and Interpretation
8. Deployment and Monitoring

- نکته:

کنار مصورسازی، عدد هم داشته باش

---

### **06-02**
<div dir='rtl'>

**1 اولین کار بعد از گرفتن داده، ببین نوعش چیه.**

انواع داده:

1. Numeric
2. String
3. Boolean
4. Date and Time
</div>

- `df.dtypes`
- `df['col'].dtype`
- `df.info()`
<div dir='rtl'>

**2 برسی صحت ستون های عددی**
- NaN داره؟
- اگر string تبدیل کن به عدد : `pd.to_numeric`
</div>

- `df.describe()`
<div dir='rtl'>

**3 بررسی مقادیر یکتا در ستون‌های دسته‌ای**
</div>

- `df.['col'].unique()`
<div dir='rtl'>

**4 شناسایی انواع داده‌های ترکیبی | Mixed Data Types**
</div>

```python
# برای تشخیص دیتافریم هایی که ستون هایی با داده های میکس دارند
def detect_mixed_types(df): # ورودی، خود دیتافریم رو میگیره
    mixed_types = {}
    for column in df.columns:
        types = df[column].apply(lambda v: type(v).__name__).value_counts()
        if len(types) > 1:
            mixed_types[column] = types.to_dict()
    return mixed_types
```

```python
# برای ستون‌هایی که باید عددی باشند اما حاوی مقادیر رشته‌ای هستند
def clean_numeric_column(series): # ورودی ستون میگیره
    # Convert to numeric, forcing errors to NaN
    numeric_series = pd.to_numeric(series, errors="coerce")

    # Check what percentage of data was lost
    percent_lost = (series.dropna().size - numeric_series.count()) / series.size * 100

    print(f"Percentage of data lost: {percent_lost:.2f}%")
    return numeric_series
```
<div dir='rtl'>

**5 انتخاب براساس نوع داده**
- مثلا اگر بخوای عملیاتی رو روی یک داده خاص انجام بدی
</div>

- `df.select_dtypes(include=[""])`
<div dir='rtl'>

**6 برسی میزان مصرف حافظه**
</div>

- `df.memory_usage()`
<div dir='rtl'>

**7 میتونی قبل از بارگزاری دیتافریم رو تغییر بدی | Ensuring Data Integrity During Loading**

**8 range دادها**
- مثال: رنج سن از یک حدی زیاد یا کم نباشه - اگر داده منفی داریم مثبتش کن

**9 بررسی رکوردهای تکراری | Checking for Duplicate Records**

**10 تأیید یکپارچگی داده‌های طبقه‌بندی‌شده**
- داده های دسته بندی شده درست دسته بندی شده اند؟
</div>

- `df["col"].isin()`
---

### **06-02**
<div dir='rtl'>

مراحل کلی Handling Missing Data:
1. انواعش چیاست؟
2. تشخیص بدیم
3. پرش کن

نکته:
- درک ماهیت Missing Data مهمتر از اینکه یک الگوریتم خاصی برای پر کردنش بنویسی

**مرحله 2**

تعداد و درصد داده گمشده در هر ستون:
</div>

```python
# Count missing values in each column
missing_counts = df.isnull().sum()

# Calculate percentage of missing values
missing_percentages = 100 * df.isnull().sum() / len(df)

# Combine counts and percentages in a single dataframe
missing_table = pd.concat(
    [missing_counts, missing_percentages],
    axis=1,
    keys=["Missing Count", "Missing Percentage"],
)
missing_table
```
<div dir='rtl'>
تصویری: (تعداد feature زیاد باشه این خوب نیست)
</div>

```python
msno.matrix(df)
```
<div dir='rtl'>

Detecting Hidden Missing Values :

بعضی وقتا مقدارگمشده دقیقا NaN نیست و شکل دیگه ای دارند. (معنیشون ناله ولی دقیقا NaN نیستند)

- پیداشون کن:
</div>

```python
# Check for suspicious values that might represent missing data
suspicious_values = ["N/A", "None", "-999", "", " "]

for column in df.columns:
    if df[column].dtype == "object":  # Only check string columns
        value_counts = df[column].value_counts()
        for value in suspicious_values:
            if value in value_counts.index:
                print(
                    f"Column '{column}' contains {value_counts[value]} instances of '{value}'"
                )
```
<div dir='rtl'>

- جایگزین کن:
</div>

```python
# Convert suspicious values to NaN
df.replace(suspicious_values, np.nan)
```
<div dir='rtl'>

درصد داده های گم شده: 
</div>

```python
def categorize_missingness(df, threshold=5):
    missing_percentages = 100 * df.isnull().sum() / len(df)

    no_missing = missing_percentages[missing_percentages == 0].index.tolist()
    low_missing = missing_percentages[
        (missing_percentages > 0) & (missing_percentages <= threshold)
    ].index.tolist()
    high_missing = missing_percentages[missing_percentages > threshold].index.tolist()

    print(f"Variables with no missing data: {no_missing}")
    print(f"Variables with low missing data (<= {threshold}%): {low_missing}")
    print(f"Variables with high missing data (> {threshold}%): {high_missing}")


categorize_missingness(df)
```
<div dir='rtl'>

**مرحله 3**

1. با میانگین یا median پر کن

2. برای دیتاهای گروه بندی با mode :
</div>

```python
df["col_mode_imputed"] = df["col"].fillna(
    df["col"].mode()[0]
)

print("Original 'col' value counts:")df
print(df["col"].value_counts(dropna=False))
print("\n'embarked' value counts after mode imputation:")
print(df["col_mode_imputed"].value_counts(dropna=False))
```
<div dir='rtl'>

3. Regression Imputation
- نکته: بعد از این کار همبستگی بین feature ها بیشتر میشه

4. K-Nearest Neighbors (KNN) Imputation

5. Time Series Imputation
</div>

- `df["col"].ffill()`
- `df["col"].bfill()`
- `df['col'].interpolate()`
<div dir='rtl'>

6. Multiple Imputation

---

### **06-03**
## Feature scaling
<div dir='rtl'>

دو تا از انواع معروفش :
</div>

- Normalization(Min-Max scaling) : کمترین مقدار را به ۰، بیشترین مقدار را به ۱، و هر مقدار دیگر را به عددی اعشاری بین ۰ و ۱ نگاشت می‌کند.
- Standardization : داده‌ها را حول عدد صفر متمرکز کرده و آن‌ها را به گونه‌ای مقیاس‌دهی می‌کند که دارای واریانس واحد باشند
<div dir='rtl'>

ویژگی های Normalization :
- دیگه مقدار صفر رو، صفر نشون نمیده برای همین برای دادهای sparse خوب نیست.
- خیلی حساس به داده های پرت. چون دقیقا min و max رو داره

نکته :
- زمانی که داده رو normaliz میکنی shap تغییری نمیکنه فقط فشرده میشه بین 0و1. (توزیعش عوض نمیشه)
</div>

- Other Data Scaling Methods    
  - Robust Scaling    
  - Max Abs Scaling   
  - Quantile Transformation    
  - Power Transformation

---